# 03 - Final Inference (HITL Remainder + Retweet Merge)

Implements **Step 4** of `classification_strategy.md`. Once the active-learning loop in
`02_hitl_training_loop.ipynb` is complete and the model quality is satisfactory, this
notebook produces the canonical labelled file `final_annotated_tweets.{pkl,csv}` covering
every tweet in the partitionable corpus **plus** every retweet — without re-classifying
retweets unnecessarily.

The pipeline runs in three passes:

1. **Pass 1 — Classify the partitionable corpus.** Apply the trained Twitter-RoBERTa to
   every non-retweet that doesn't already have a human or LLM-bootstrap label.
2. **Pass 2 — Promote orphan originals.** For each retweet whose referenced original is
   missing from the labelled set, group orphans by `ref_id`, pick one representative per
   group, classify it once, and store the prediction keyed by the missing original's id.
3. **Pass 3 — Look up retweet labels.** Every retweet inherits its label from the
   labelled corpus or from the orphan-originals dict via `referenced_tweets_dictionary['id']`.
   Retweets with no usable reference are caught by a per-row model fallback.

Output carries a `label_source` provenance column so downstream analysis can audit how
each label was produced (`human`, `llm_bootstrap`, `model_original`, `lookup`,
`model_synthetic_retweet`, `model_no_reference`).

In [ ]:
%%time
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → AItrust_twits_pruned_dict.json    → Partitioned Data/AI Data/
# 'Art' → AItrust_Art_pruned_twit_dict.json → Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
partitioned_folder = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
hitl_folder        = datasets_folder / 'Classifiers_Data' / 'HITL'
final_folder       = datasets_folder / 'Classifiers_Data' / 'Final'
final_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
%%time
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'torch', 'tqdm'])
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
%%time
import time
import numpy as np
import pandas as pd
import torch
import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

## Configuration

In [ ]:
%%time
# Larger ROBERTA_MULTI multiplies the pipeline batch_size for inference. Keep
# ROBERTA_BATCH conservative and let ROBERTA_MULTI grow on bigger GPUs.
ROBERTA_BATCH = 64
ROBERTA_MULTI = 8     # outer chunk fed to clf_pipe per call = ROBERTA_BATCH * ROBERTA_MULTI
OUTER_BATCH   = ROBERTA_BATCH * ROBERTA_MULTI

# Paths sourced from the partitioning notebook (00) and the LLM/HITL labelling steps.
PARTITIONED_PICKLES = [
    'llm_bootstrap_dataset.pkl',
    'base_dataset.pkl',
    'inference_dataset.pkl',
]
RETWEETS_PICKLE = 'retweets_dataset.pkl'
LLM_LABELS_CSV  = hitl_folder / 'llm_bootstrap_labels.csv'

## 1. Load the trained model

In [ ]:
%%time
best_path = classifiers_folder / 'best_roberta_model'
if not best_path.exists():
    raise FileNotFoundError(f'Model not found at {best_path}. Run notebook 02 first.')

tokenizer = AutoTokenizer.from_pretrained(str(best_path))
model     = AutoModelForSequenceClassification.from_pretrained(str(best_path))
device    = 0 if torch.cuda.is_available() else -1
clf_pipe  = pipeline(
    'text-classification', model=model, tokenizer=tokenizer,
    device=device, batch_size=ROBERTA_BATCH, return_all_scores=True,
    truncation=True, max_length=128,
)
print(f'Model loaded. Device: {"GPU" if device == 0 else "CPU"}')

def predict_top(texts):
    """Run clf_pipe over an iterable of texts and yield (label, confidence) per row."""
    preds = []
    for i in range(0, len(texts), OUTER_BATCH):
        for scores in clf_pipe(texts[i:i + OUTER_BATCH]):
            top = max(scores, key=lambda x: x['score'])
            preds.append((top['label'], float(top['score'])))
    return preds

## 2. Build the labels dict from existing labelled CSVs

Sources of labels that already exist before Pass 1:
- `llm_bootstrap_labels.csv` — Path A LLM seed (Step 0 of the strategy doc).
- `hitl_review_batch_*.csv` — every round of human review, including the optional Path B
  seed at `hitl_review_batch_00.csv`.

Schema per `labels` entry: `{label, confidence, label_source}` keyed by tweet id (str).

In [ ]:
%%time
labels: dict[str, dict] = {}

# 2a. LLM bootstrap labels (Path A). PARSE_ERROR rows are dropped.
if LLM_LABELS_CSV.exists():
    llm_df = pd.read_csv(LLM_LABELS_CSV)
    llm_df = llm_df[
        llm_df['predicted_label'].notna() &
        (llm_df['predicted_label'].astype(str) != 'PARSE_ERROR')
    ]
    has_conf = 'confidence' in llm_df.columns
    for row in llm_df.itertuples(index=False):
        labels[str(row.id)] = {
            'label':        str(row.predicted_label),
            'confidence':   float(row.confidence) if has_conf else 1.0,
            'label_source': 'llm_bootstrap',
        }
    print(f'LLM bootstrap labels loaded: {len(llm_df):,}')
else:
    print(f'No LLM bootstrap labels at {LLM_LABELS_CSV} — Path A not used.')

# 2b. HITL human labels (overrides LLM bootstrap if any tweet was double-labelled,
#     which should not happen given the partitions are disjoint).
n_human = 0
for csv_path in sorted(hitl_folder.glob('hitl_review_batch_*.csv')):
    review_df = pd.read_csv(csv_path)
    if 'human_label' not in review_df.columns:
        continue
    labelled = review_df[
        review_df['human_label'].notna() &
        (review_df['human_label'].astype(str).str.strip() != '')
    ]
    for row in labelled.itertuples(index=False):
        labels[str(row.id)] = {
            'label':        str(row.human_label),
            'confidence':   1.0,
            'label_source': 'human',
        }
    n_human += len(labelled)
    print(f'  {csv_path.name}: {len(labelled):,} human labels')

n_llm = sum(1 for v in labels.values() if v['label_source'] == 'llm_bootstrap')
print(f'Labels dict bootstrap: {len(labels):,} '
      f'({n_human:,} human, {n_llm:,} LLM)')

## 3. Pass 1 — Classify the unlabelled non-retweets

Loads every non-retweet partition pickle, identifies rows that don't already have a
label from Step 2, and runs RoBERTa on those. The result populates `labels` with
`label_source='model_original'` for every newly classified tweet.

In [ ]:
%%time
# Load every non-retweet partition pickle (LLM bootstrap, base, HITL batches, inference).
frames = []
for name in PARTITIONED_PICKLES:
    p = partitioned_folder / name
    if not p.exists():
        print(f'WARNING: {p} missing — skipping.')
        continue
    frames.append(pd.read_pickle(p))
for p in sorted(partitioned_folder.glob('hitl_pending_batch_*.pkl')):
    frames.append(pd.read_pickle(p))

if not frames:
    raise FileNotFoundError(f'No partition pickles found in {partitioned_folder}.')

non_retweets = pd.concat(frames, ignore_index=True)
non_retweets['id'] = non_retweets['id'].astype(str)
# Defensive de-duplication (partitions should be disjoint by construction).
non_retweets = non_retweets.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
print(f'Non-retweet partitionable corpus: {len(non_retweets):,}')

mask_unlabelled = ~non_retweets['id'].isin(labels)
to_classify = non_retweets[mask_unlabelled].reset_index(drop=True)
print(f'Already labelled (skip):  {(~mask_unlabelled).sum():,}')
print(f'To classify with RoBERTa: {len(to_classify):,}')

if len(to_classify):
    t0 = time.time()
    preds = predict_top(to_classify['text'].astype(str).tolist())
    print(f'Pass 1 RoBERTa: {time.time() - t0:.1f}s on {len(preds):,} rows')
    for tid, (label, conf) in zip(to_classify['id'].tolist(), preds):
        labels[tid] = {
            'label':        label,
            'confidence':   conf,
            'label_source': 'model_original',
        }

print(f'Labels dict after Pass 1: {len(labels):,}')

## 4. Pass 2 — Promote orphan originals

An **orphan retweet** is one whose `referenced_tweets_dictionary['id']` is not a key in
`labels` — its referenced original is missing from the partitioned corpus. We group
orphans by `ref_id`, pick the highest-engagement copy as the representative (ties broken
by lowest `id` for determinism), classify only the representative, and write the result
into `orphan_originals` keyed by the missing original's id. All sibling retweets later
inherit this label by lookup in Pass 3.

In [ ]:
%%time
retweets_path = partitioned_folder / RETWEETS_PICKLE
if not retweets_path.exists():
    raise FileNotFoundError(
        f'{retweets_path} not found. Re-run 00_hitl_data_preparation.ipynb.'
    )
retweets_df = pd.read_pickle(retweets_path)
retweets_df['id'] = retweets_df['id'].astype(str)
print(f'Retweets to merge: {len(retweets_df):,}')

def _ref_id(rd):
    if isinstance(rd, dict):
        rid = rd.get('id')
        if rid is not None:
            return str(rid)
    return None

# Prefer the precomputed `ref_id` column (written by 00 since the RAM fix).
# Fall back to extracting from `referenced_tweets_dictionary` for older partitions.
if 'ref_id' in retweets_df.columns:
    retweets_df['ref_id'] = retweets_df['ref_id'].where(retweets_df['ref_id'].notna(), None)
elif 'referenced_tweets_dictionary' in retweets_df.columns:
    retweets_df['ref_id'] = retweets_df['referenced_tweets_dictionary'].apply(_ref_id)
else:
    print('WARNING: retweets_dataset.pkl has no ref_id / referenced_tweets_dictionary column.')
    print('         Every retweet will fall through to the no-reference fallback in Pass 3.')
    retweets_df['ref_id'] = None

n_no_ref = retweets_df['ref_id'].isna().sum()
rate_no_ref = n_no_ref / max(len(retweets_df), 1)
print(f'Retweets with no usable ref_id: {n_no_ref:,} ({rate_no_ref:.1%})')
if rate_no_ref > 0.01:
    print('  > 1% — investigate referenced_tweets parsing in 02_Processing/02 before trusting Pass 3.')

ref_in_labels = retweets_df['ref_id'].isin(labels)
orphan_mask   = retweets_df['ref_id'].notna() & ~ref_in_labels
orphans       = retweets_df[orphan_mask]
print(f'Standard-lookup retweets: {ref_in_labels.sum():,}')
print(f'Orphan retweets:          {len(orphans):,}')

orphan_originals: dict[str, dict] = {}
if len(orphans):
    sorted_orphans = (orphans
                      .assign(_eng=lambda d: d['likes'] + d['retweets'])
                      .sort_values(['_eng', 'id'], ascending=[False, True]))
    representatives = sorted_orphans.drop_duplicates(subset='ref_id', keep='first')
    print(f'Distinct missing originals (one model call each): {len(representatives):,}')

    t0 = time.time()
    rep_preds = predict_top(representatives['text'].astype(str).tolist())
    print(f'Pass 2 RoBERTa: {time.time() - t0:.1f}s')

    for ref_id, (label, conf) in zip(representatives['ref_id'].tolist(), rep_preds):
        orphan_originals[ref_id] = {
            'label':        label,
            'confidence':   conf,
            'label_source': 'model_synthetic_retweet',
        }

print(f'orphan_originals dict size: {len(orphan_originals):,}')

# Invariant: a ref_id should never be in BOTH labels and orphan_originals.
shared = set(labels.keys()) & set(orphan_originals.keys())
assert not shared, f'Invariant violated: {len(shared)} ref_ids in both dicts'

## 5. Pass 3 — Look up retweet labels

For every retweet:
1. If `ref_id` is in `labels` (a real labelled original/reply/quote), inherit that label
   with `label_source='lookup'`.
2. Else if `ref_id` is in `orphan_originals` (a synthetic original we promoted in Pass 2),
   inherit that label with `label_source='lookup'` — the synthetic provenance is
   already recorded on the synthetic original itself.
3. Else (no usable `ref_id`), classify the retweet directly with
   `label_source='model_no_reference'`.

In [ ]:
%%time
retweet_records = []
no_ref_rows    = []

for row in retweets_df.itertuples(index=False):
    ref_id = row.ref_id
    if ref_id in labels:
        rec = labels[ref_id]
    elif ref_id in orphan_originals:
        rec = orphan_originals[ref_id]
    else:
        no_ref_rows.append((row.id, str(row.text)))
        continue
    retweet_records.append({
        'id':              row.id,
        'predicted_label': rec['label'],
        'confidence':      rec['confidence'],
        'label_source':    'lookup',
    })

if no_ref_rows:
    print(f'No-reference retweets to classify directly: {len(no_ref_rows):,}')
    t0 = time.time()
    no_ref_preds = predict_top([t for _, t in no_ref_rows])
    print(f'Pass 3 fallback RoBERTa: {time.time() - t0:.1f}s')
    for (tid, _t), (label, conf) in zip(no_ref_rows, no_ref_preds):
        retweet_records.append({
            'id':              tid,
            'predicted_label': label,
            'confidence':      conf,
            'label_source':    'model_no_reference',
        })

retweet_label_df = pd.DataFrame(retweet_records).set_index('id')
print(f'Retweet labels assembled: {len(retweet_label_df):,}')

## 6. Assemble + save the final annotated file

Joins partition metadata (text, processed_text, type, likes, retweets) onto the
label/confidence/source produced by the three passes and writes a single annotated
table to `Classifiers_Data/Final/`.

In [ ]:
%%time
# Non-retweet metadata + labels
non_retweet_label_df = (pd.DataFrame.from_dict(labels, orient='index')
                         .rename(columns={'label': 'predicted_label'}))
non_retweet_label_df.index.name = 'id'

non_retweet_final = non_retweets.set_index('id').join(non_retweet_label_df, how='left')

# Retweet metadata + labels (drop placeholder predicted_label/human_label columns from 00)
drop_cols = [c for c in ('predicted_label', 'human_label', 'ref_id')
             if c in retweets_df.columns]
retweet_meta = retweets_df.drop(columns=drop_cols).set_index('id')
retweet_final = retweet_meta.join(retweet_label_df, how='left')

final_df = pd.concat([non_retweet_final, retweet_final], axis=0).reset_index()

preferred = ['id', 'text', 'processed_text', 'type', 'likes', 'retweets',
             'predicted_label', 'confidence', 'label_source']
cols = [c for c in preferred if c in final_df.columns] + \
       [c for c in final_df.columns if c not in preferred]
final_df = final_df[cols]

print(f'Final annotated dataset: {len(final_df):,} rows')
print()
print('label_source distribution:')
print(final_df['label_source'].value_counts(dropna=False).to_string())
print()
print('predicted_label distribution (top 20):')
print(final_df['predicted_label'].value_counts(dropna=False).head(20).to_string())

out_pkl = final_folder / 'final_annotated_tweets.pkl'
out_csv = final_folder / 'final_annotated_tweets.csv'
final_df.to_pickle(out_pkl)
final_df.to_csv(out_csv, index=False)
print(f'\nSaved → {out_pkl}')
print(f'Saved → {out_csv}')

In [ ]:
# Disconnect from Colab runtime (no-op locally)
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    pass
